# 15 — Sintéticos I: generadores de escenarios con régimen conocido

> **Estado: ESQUELETO.** Este notebook fija el diseño antes de implementar nada. Todas las secciones de trabajo son celdas `TODO` controladas por el flag `IMPLEMENTADO` (por defecto `False`): con `False` el notebook se ejecuta de principio a fin sin error y solo informa de lo pendiente; con `True` cada `TODO` sin implementar levanta `NotImplementedError`. Ninguna cifra de este texto está escrita a mano: las que aparezcan saldrán de celdas.

## 0. Objetivo

Construir generadores de trayectorias multivariantes de las features del benchmark **con la
secuencia de régimen conocida** (verdad-terreno exacta), ajustados solo con el tramo de
entrenamiento, para (a) validar su fidelidad en `16`, (b) estresar los detectores en un
laboratorio controlado en `17` y (c) ampliar la escasa muestra de crisis en `18`.

## Preguntas que responde

1. ¿Qué generadores paramétricos reproducen la dinámica de régimen del tramo de entrenamiento
   (duraciones, transiciones, niveles de volatilidad y correlación por régimen)?
2. ¿Aportan los generadores neuronales realismo adicional (colas, dependencia no lineal,
   asimetrías) sin memorizar el entrenamiento?
3. ¿Cómo se impone una secuencia de régimen dada (condicionamiento) y cómo se etiqueta la
   verdad-terreno en la salida?
4. ¿Cuál es el coste de ajuste/muestreo y es reproducible con la semilla del proyecto?

## Entradas y salidas (vía `regimenes.rutas`)

| papel | ruta |
|---|---|
| paneles reales por pista | `rutas.DATA_PROCESSED` |
| banco de pruebas (ventanas, crisis, drawdowns) | `rutas.BENCHMARK_SPEC` |
| configuración de generadores | `rutas.SINTETICOS_CONFIG` |
| trayectorias generadas (gitignored) | `rutas.DATA_SINTETICOS / <generador> / <pista>` |
| resumen de ajuste, parámetros y tiempos | `rutas.RESULTS_SINTETICOS / 'generadores'` |

## Metodología prevista

Todos los generadores implementan la interfaz `regimenes.sinteticos.Generador`
(`name`, `fit(train, regimes)`, `sample(n_paths, length, regimes, random_state)` → lista de
`DataFrame` con las columnas de `train` más `regime`) y se registran en
`regimenes.sinteticos.registry`.

**Paramétricos** (`regimenes.sinteticos.parametricos`):

- **Bootstrap por bloques condicionado a régimen.** Se parte el entrenamiento por régimen de
  referencia, se remuestrean bloques (bootstrap estacionario de longitud media $\ell$) *dentro*
  de cada régimen y se encadenan siguiendo una cadena de Markov con la matriz de transición
  estimada $\hat P$. Conserva marginales y dependencia intra-bloque; no crea escenarios fuera de
  lo observado (Künsch 1989; Politis y Romano 1994).
- **MS-VAR.** $y_t = \nu_{s_t} + A_{s_t} y_{t-1} + \Sigma_{s_t}^{1/2}\varepsilon_t$, con $s_t$
  Markov de matriz $P$; ajuste EM y simulación directa (`hamilton1989`; Krolzig 1997). Reutiliza
  la especificación de D05.
- **HMM con emisiones t multivariantes.** $y_t \mid s_t=k \sim t_{\nu_k}(\mu_k, \Sigma_k)$; colas
  pesadas por régimen, simulación directa de la cadena (`hmm_bullabulla2006`). Reutiliza D08.
- **GARCH con régimen.** $\sigma^2_t = \omega_{s_t} + \alpha_{s_t}\varepsilon^2_{t-1} +
  \beta_{s_t}\sigma^2_{t-1}$ con innovaciones t (`vol_gray1996`, `vol_haasmittnikpaolella2004`)
  para el factor de mercado; el resto de features, por regresión sobre el factor o cópula por
  régimen (decisión abierta, se documentará aquí).

**Neuronales** (`regimenes.sinteticos.neuronales`, extra `[deep]`):

- **Modelos del repositorio de clase del usuario** (ruta y nombres por confirmar): se adaptan
  a la interfaz `Generador` sin cambiar su arquitectura.
- **TimeGAN** (Yoon, Jarrett y van der Schaar 2019): red de *embedding*, supervisor y juego
  adversarial en el espacio latente; condicionamiento por etiqueta de régimen concatenada.
- **VAE temporal** (`nn_kingma2014`; variante tipo TimeVAE, Desai et al. 2021) con decodificador
  condicionado a régimen.
- **QuantGAN** (Wiese et al. 2020): generador TCN para rentabilidades de colas pesadas; se
  extiende a condicionamiento por régimen.

Referencias sin clave aún en `docs/references.bib` (pendientes de añadir): Künsch (1989),
Politis y Romano (1994), Krolzig (1997), Yoon et al. (2019), Desai et al. (2021), Wiese et al. (2020).

## Riesgos

- **Fuga temporal.** Todo generador se ajusta **solo** con el tramo de entrenamiento
  (`entrenamiento.fin_train` de `configs/sinteticos.yaml`); normalizaciones, selección de
  hiperparámetros y etiquetas de régimen de referencia también se calculan solo con ese tramo.
  Mientras `fin_train` sea `null` no se puede pasar `IMPLEMENTADO=True` (la celda de
  configuración lo comprueba).
- **Circularidad.** Si las etiquetas de régimen con las que se condiciona el generador salen de un
  detector del benchmark, ese detector (y su familia) juega en casa al evaluarse sobre lo
  generado. Se mitiga con etiquetas de referencia independientes de los detectores (ventanas de
  crisis y suelos de drawdown de `configs/benchmark_spec.yaml`, o varias etiquetas de familias
  distintas) y leyendo la matriz generador × detector con la diagonal "misma familia" aparte.
- **Identificación del régimen de crisis.** Hay pocos episodios de crisis en el entrenamiento:
  los parámetros del régimen de crisis estarán mal identificados; se reportará su incertidumbre.
- **Memorización** (neuronales): se controla en `16` con la distancia al vecino real más cercano.

**Índice**

0. Objetivo, preguntas, entradas/salidas, metodología y riesgos
1. Configuración
2. Datos de entrenamiento y régimen de referencia
3. Generadores paramétricos
4. Generadores neuronales
5. Muestreo y persistencia de trayectorias
6. Resumen y siguiente paso


## 1. Configuración

In [1]:
# Configuración común del esqueleto (no lee results/benchmark ni ajusta nada).
import yaml
import pandas as pd
from IPython.display import display

from regimenes import rutas
from regimenes import sinteticos
from regimenes.sinteticos import Generador
from regimenes.sinteticos import registry as registro_sinteticos

IMPLEMENTADO = False  # pasar a True solo cuando las secciones TODO estén implementadas

CONFIG_SINTETICOS = yaml.safe_load(rutas.SINTETICOS_CONFIG.read_text(encoding='utf-8'))
SEMILLA = CONFIG_SINTETICOS.get('semilla')


def saltar(seccion: str, detalle: str) -> None:
    """Punto TODO controlado: informa y sigue (solo se llama con IMPLEMENTADO=False)."""
    print(f'[TODO pendiente] {seccion}: {detalle}')


def tabla_rutas(rutas_nb: dict) -> pd.DataFrame:
    """Entradas/salidas del notebook (siempre vía regimenes.rutas) y si existen ya en disco."""
    filas = [
        {'papel': papel, 'ruta': ruta.relative_to(rutas.ROOT).as_posix(), 'existe': ruta.exists()}
        for papel, ruta in rutas_nb.items()
    ]
    return pd.DataFrame(filas).set_index('papel')

RUTAS_NB = {
    'entrada: paneles reales': rutas.DATA_PROCESSED,
    'entrada: banco de pruebas': rutas.BENCHMARK_SPEC,
    'entrada: configuración sintéticos': rutas.SINTETICOS_CONFIG,
    'salida: trayectorias': rutas.DATA_SINTETICOS,
    'salida: resumen de generadores': rutas.RESULTS_SINTETICOS / 'generadores',
}
display(tabla_rutas(RUTAS_NB))
print('Semilla:', SEMILLA, '| pista de entrenamiento:', CONFIG_SINTETICOS['entrenamiento'].get('pista'))
print('Generadores registrados:', sorted(registro_sinteticos.GENERADORES) or 'ninguno todavía')
print('Generadores declarados en la configuración:', sorted(CONFIG_SINTETICOS.get('generadores') or {}) or 'ninguno todavía')


,ruta,existe
papel,,
entrada: paneles reales,data/processed,True
entrada: banco de pruebas,configs/benchmark_spec.yaml,True
entrada: configuración sintéticos,configs/sinteticos.yaml,True
salida: trayectorias,data/sinteticos,True
salida: resumen de generadores,results/sinteticos/generadores,False


Semilla: 42 | pista de entrenamiento: B
Generadores registrados: ninguno todavía
Generadores declarados en la configuración: ninguno todavía


In [2]:
fin_train = CONFIG_SINTETICOS['entrenamiento'].get('fin_train')
if fin_train is None:
    mensaje = ('configs/sinteticos.yaml: entrenamiento.fin_train = null. Fijar el corte de '
               'entrenamiento antes de ajustar cualquier generador (regla anti-fuga).')
    if IMPLEMENTADO:
        raise ValueError(mensaje)
    print('[aviso]', mensaje)
else:
    print('Corte de entrenamiento de los generadores:', fin_train)


[aviso] configs/sinteticos.yaml: entrenamiento.fin_train = null. Fijar el corte de entrenamiento antes de ajustar cualquier generador (regla anti-fuga).


## 2. Datos de entrenamiento y régimen de referencia

Se carga el panel de la pista indicada en la configuración, se recorta en `fin_train` y se construye la etiqueta de régimen de referencia **independiente de los detectores** (ventanas de crisis del banco de pruebas; alternativa documentada si se usa otra).

In [3]:
# TODO 15.2 — panel de entrenamiento y régimen de referencia
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name
from regimenes.benchmark import load_benchmark_spec

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. panel = pd.read_parquet(rutas.DATA_PROCESSED / f'pista{pista}...') recortado a <= fin_train
    # 2. features = CONFIG_SINTETICOS['entrenamiento']['features'] o el núcleo de la pista
    # 3. regimen_ref = etiqueta 0/1 desde crisis_windows de configs/benchmark_spec.yaml (sin detectores)
    # 4. tabla de duraciones y matriz de transición empírica del régimen de referencia (solo train)
    raise NotImplementedError('TODO 15.2: cargar el panel de train y construir el régimen de referencia')
else:
    saltar('15.2', 'panel de entrenamiento y régimen de referencia')


[TODO pendiente] 15.2: panel de entrenamiento y régimen de referencia


## 3. Generadores paramétricos

Una celda por generador: implementación en `regimenes.sinteticos.parametricos`, registro y ajuste solo con el tramo de entrenamiento.

In [4]:
# TODO 15.3a — bootstrap por bloques condicionado a régimen
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. class BootstrapRegimen(Generador) en regimenes.sinteticos.parametricos
    # 2. fit: bloques por régimen + matriz de transición P estimada en train
    # 3. sample: simula la cadena (o usa la impuesta) y rellena con bloques del régimen
    # 4. longitud media de bloque desde CONFIG_SINTETICOS['generadores'][...]['params']
    raise NotImplementedError('TODO 15.3a: implementar BootstrapRegimen(Generador)')
else:
    saltar('15.3a', 'bootstrap por bloques condicionado a régimen')


[TODO pendiente] 15.3a: bootstrap por bloques condicionado a régimen


In [5]:
# TODO 15.3b — MS-VAR
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. class MSVAR(Generador): reutiliza la especificación de D05 (regimenes.detectores.f4_switching)
    # 2. fit por EM en train; sample por simulación directa de s_t y del VAR por régimen
    raise NotImplementedError('TODO 15.3b: implementar MSVAR(Generador)')
else:
    saltar('15.3b', 'MS-VAR')


[TODO pendiente] 15.3b: MS-VAR


In [6]:
# TODO 15.3c — HMM con emisiones t
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. class HMMt(Generador): reutiliza el ajuste de D08 (regimenes.detectores.f3_hmm)
    # 2. sample: cadena de Markov + emisiones t multivariantes por estado
    raise NotImplementedError('TODO 15.3c: implementar HMMt(Generador)')
else:
    saltar('15.3c', 'HMM con emisiones t')


[TODO pendiente] 15.3c: HMM con emisiones t


In [7]:
# TODO 15.3d — GARCH con régimen
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. class GARCHRegimen(Generador): GJR-GARCH-t por régimen para el factor de mercado (arch)
    # 2. resto de features: regresión sobre el factor o cópula por régimen (decisión abierta)
    raise NotImplementedError('TODO 15.3d: implementar GARCHRegimen(Generador)')
else:
    saltar('15.3d', 'GARCH con régimen')


[TODO pendiente] 15.3d: GARCH con régimen


## 4. Generadores neuronales

Requieren el extra `[deep]` (torch). Mismo contrato `Generador`; se entrenan solo con el tramo de entrenamiento y con semilla fija.

In [8]:
# TODO 15.4a — modelos del repositorio de clase
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. localizar el repositorio de clase del usuario (ruta por confirmar) y sus modelos
    # 2. envolver cada modelo en una subclase de Generador en regimenes.sinteticos.neuronales
    raise NotImplementedError('TODO 15.4a: adaptar los modelos del repositorio de clase a Generador')
else:
    saltar('15.4a', 'modelos del repositorio de clase')


[TODO pendiente] 15.4a: modelos del repositorio de clase


In [9]:
# TODO 15.4b — TimeGAN condicionado a régimen
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. class TimeGANRegimen(Generador): embedding + supervisor + adversarial; etiqueta de régimen concatenada
    raise NotImplementedError('TODO 15.4b: implementar TimeGANRegimen(Generador)')
else:
    saltar('15.4b', 'TimeGAN condicionado a régimen')


[TODO pendiente] 15.4b: TimeGAN condicionado a régimen


In [10]:
# TODO 15.4c — VAE temporal condicionado
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. class VAERegimen(Generador): codificador recurrente, decodificador condicionado a régimen
    raise NotImplementedError('TODO 15.4c: implementar VAERegimen(Generador)')
else:
    saltar('15.4c', 'VAE temporal condicionado')


[TODO pendiente] 15.4c: VAE temporal condicionado


In [11]:
# TODO 15.4d — QuantGAN
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. class QuantGANRegimen(Generador): generador TCN para rentabilidades, extendido con régimen
    raise NotImplementedError('TODO 15.4d: implementar QuantGANRegimen(Generador)')
else:
    saltar('15.4d', 'QuantGAN')


[TODO pendiente] 15.4d: QuantGAN


## 5. Muestreo y persistencia de trayectorias

Para cada generador registrado: `sample(n_paths, length, regimes, random_state=SEMILLA)` con los parámetros de `muestreo` de la configuración; se guardan las trayectorias en `rutas.DATA_SINTETICOS` y un resumen (tiempos, parámetros, huella de configuración) en `rutas.RESULTS_SINTETICOS / 'generadores'`.

In [12]:
# TODO 15.5 — muestreo y guardado
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. for nombre in registro_sinteticos.GENERADORES: gen = registro_sinteticos.crear(nombre, **params)
    # 2. gen.fit(train, regimen_ref); trayectorias = gen.sample(n_paths, length, regimes, random_state=SEMILLA)
    # 3. guardar en rutas.DATA_SINTETICOS / nombre / pista (parquet) + resumen csv en rutas.RESULTS_SINTETICOS
    raise NotImplementedError('TODO 15.5: muestrear y persistir trayectorias')
else:
    saltar('15.5', 'muestreo y guardado')


[TODO pendiente] 15.5: muestreo y guardado


## 6. Resumen y siguiente paso

Pendiente de implementación. Las conclusiones se escribirán con cifras impresas por celdas. Siguiente: `16_sinteticos_validacion`.